# 取引量調整・確率校正

歴史的な実験コードです。現在の実行入口は `../09_confidence_nested.ipynb`。
元Notebookのセル番号は0始まりです。コードの個人フォルダ名は置換しています。独立実行は保証しません。
保存出力は `../../results/imported_20260907/`、監査は `../../docs/CONFIDENCE_AUDIT.md` を参照してください。


## 元のセル index 22


In [ ]:
# ============================================================
# CONFIDENCE SIZING + TP/SL + WALK-FORWARD
#
# 目的
# 1. P(up), P(down) が高いほど本当に利益が増えるか
# 2. Confidence別にポジションサイズを変えると改善するか
# 3. BUY/SELL別にTP/SLを選ぶと改善するか
# 4. 最終的にPF, DD, Sharpe, Growthが改善するか
#
# 前提:
# 以前のセルで以下が定義済み
#
# load_bars
# prepare_data
# outer_folds
# fit_base_models
# predict_base_models
# simulate_trade
# strategy_stats
# HORIZON_BARS
# ============================================================


from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. 実験設定
# ============================================================

# BUY/SELLの最低Confidence
CONFIDENCE_THRESHOLDS = [
    0.52,
    0.55,
    0.58,
    0.60,
    0.62,
    0.65,
    0.70,
    0.75,
]

# Confidenceに応じたBET倍率候補
# 1.0 = 基準BET
# 0.5 = 半分
# 1.5 = 1.5倍
BET_SIZES = [
    0.25,
    0.50,
    0.75,
    1.00,
    1.50,
]

# TP
TP_VALUES = [
    0.0003,
    0.0005,
    0.0008,
    0.0010,
    0.0015,
]

# SL
SL_VALUES = [
    0.0003,
    0.0005,
    0.0007,
    0.0010,
    0.0015,
]

# Validation最低取引数
MIN_VALIDATION_TRADES = 20

# 高Confidence判定
HIGH_CONF_LEVELS = [
    0.65,
    0.70,
    0.75,
]

# 初期資金
START_CAPITAL = 10000


# ============================================================
# 2. 最新CSV取得
# ============================================================

def find_latest_snapshot():

    root = (
        Path.cwd()
        / "fx_experiment_runs"
    )

    files = list(
        root.glob(
            "*/usdjpy_5m.csv"
        )
    )

    if not files:
        raise FileNotFoundError(
            "usdjpy_5m.csv が見つかりません"
        )

    return max(
        files,
        key=lambda p:
            p.stat().st_mtime
    )


csv_path = find_latest_snapshot()

print("使用CSV:")
print(csv_path)


# ============================================================
# 3. データ
# ============================================================

bars = load_bars(
    csv_path
)

data = prepare_data(
    bars
)

print()
print("5分足:", len(bars))
print("使用可能データ:", len(data))


# ============================================================
# 4. Confidence
#
# BUY confidence = p_up
# SELL confidence = p_down
# ============================================================

def make_confidence_signal(
    p_move,
    p_up,
    p_down,
    move_threshold,
    buy_threshold,
    sell_threshold,
):

    signals = np.zeros(
        len(p_move)
    )

    buy = (
        (p_move >= move_threshold)
        &
        (p_up >= buy_threshold)
        &
        (p_up > p_down)
    )

    sell = (
        (p_move >= move_threshold)
        &
        (p_down >= sell_threshold)
        &
        (p_down > p_up)
    )

    signals[buy] = 1
    signals[sell] = -1

    confidence = np.maximum(
        p_up,
        p_down
    )

    return (
        signals,
        confidence
    )


# ============================================================
# 5. Confidence別BET倍率
# ============================================================

def position_size_from_confidence(
    confidence,
    low_threshold,
    high_threshold,
    low_bet,
    high_bet,
):

    size = np.zeros(
        len(confidence)
    )

    medium = (
        (confidence >= low_threshold)
        &
        (confidence < high_threshold)
    )

    high = (
        confidence >= high_threshold
    )

    size[medium] = low_bet
    size[high] = high_bet

    return size


# ============================================================
# 6. Position size対応バックテスト
# ============================================================

def run_sized_backtest(
    bars,
    frame,
    signals,
    sizes,
    tp,
    sl,
    *,
    end_time=None,
):

    records = []

    next_signal_position = -1

    for (
        time,
        signal,
        size,
    ) in zip(
        frame.index,
        signals,
        sizes,
    ):

        if (
            signal == 0
            or
            size <= 0
        ):
            continue

        position = (
            bars.index
            .get_loc(
                time
            )
        )

        if (
            position
            <
            next_signal_position
        ):
            continue

        trade = simulate_trade(
            bars,
            time,
            (
                "BUY"
                if signal == 1
                else "SELL"
            ),
            tp,
            sl,
            end_time=end_time,
        )

        if trade is None:
            continue

        trade["position_size"] = (
            size
        )

        trade["sized_return"] = (
            trade["net_return"]
            *
            size
        )

        records.append(
            trade
        )

        next_signal_position = (
            position
            +
            HORIZON_BARS
        )

    return pd.DataFrame(
        records
    )


# ============================================================
# 7. 資産曲線指標
# ============================================================

def sized_strategy_stats(
    trades,
):

    if (
        trades is None
        or
        trades.empty
    ):

        return {
            "trades":
                0,

            "win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "profit_factor":
                np.nan,

            "max_dd":
                np.nan,

            "total_growth":
                0.0,

            "sharpe":
                np.nan,

            "end_capital":
                START_CAPITAL,
        }

    r = (
        trades[
            "sized_return"
        ]
        .to_numpy()
    )

    base_stats = strategy_stats(
        r
    )

    if (
        len(r) > 1
        and
        np.std(r) > 0
    ):

        sharpe = (
            np.mean(r)
            /
            np.std(r)
            *
            np.sqrt(
                len(r)
            )
        )

    else:

        sharpe = np.nan

    end_capital = (
        START_CAPITAL
        *
        np.prod(
            1 + r
        )
    )

    base_stats[
        "sharpe"
    ] = (
        sharpe
    )

    base_stats[
        "end_capital"
    ] = (
        end_capital
    )

    return base_stats


# ============================================================
# 8. Calibration診断
# ============================================================

def confidence_calibration_table(
    p_up,
    p_down,
    future_return,
):

    confidence = np.maximum(
        p_up,
        p_down
    )

    predicted_up = (
        p_up >= p_down
    )

    actual_correct = np.where(
        predicted_up,
        future_return > 0,
        future_return < 0
    )

    bins = [
        0.50,
        0.55,
        0.60,
        0.65,
        0.70,
        0.75,
        0.80,
        0.90,
        1.01,
    ]

    labels = [
        "50-55",
        "55-60",
        "60-65",
        "65-70",
        "70-75",
        "75-80",
        "80-90",
        "90-100",
    ]

    groups = pd.cut(
        confidence,
        bins=bins,
        labels=labels,
        right=False,
    )

    temp = pd.DataFrame(
        {
            "confidence":
                confidence,

            "correct":
                actual_correct,

            "future_return":
                future_return,

            "group":
                groups,
        }
    )

    result = (
        temp
        .groupby(
            "group",
            observed=True
        )
        .agg(
            count=(
                "correct",
                "size"
            ),

            mean_confidence=(
                "confidence",
                "mean"
            ),

            actual_accuracy=(
                "correct",
                "mean"
            ),

            avg_future_return=(
                "future_return",
                "mean"
            ),
        )
        .reset_index()
    )

    return result


# ============================================================
# 9. Validationで利益設定を選択
#
# 今回の目的は「利益ベース」
#
# score:
# avg return
# × sqrt(trades)
# × drawdown penalty
# ============================================================

def choose_profit_setting(
    bars,
    validation,
    probabilities,
):

    p_move, p_up, p_down = probabilities

    val_end = (
        validation.index[-1]
        +
        pd.Timedelta(
            minutes=5
        )
    )

    best = None
    best_score = -np.inf

    # 計算量を抑えるため、
    # BUY/SELL最低閾値は共通候補から選ぶ
    for (
        move_threshold,
        low_threshold,
        high_threshold,
        low_bet,
        high_bet,
        tp,
        sl,
    ) in itertools.product(

        [0.55, 0.60, 0.65, 0.70],

        CONFIDENCE_THRESHOLDS,

        HIGH_CONF_LEVELS,

        [
            0.25,
            0.50,
            0.75,
        ],

        [
            0.75,
            1.00,
            1.50,
        ],

        TP_VALUES,

        SL_VALUES,
    ):

        if (
            high_threshold
            <= low_threshold
        ):
            continue

        if (
            high_bet
            < low_bet
        ):
            continue

        (
            signals,
            confidence,
        ) = (
            make_confidence_signal(
                p_move,
                p_up,
                p_down,
                move_threshold,
                low_threshold,
                low_threshold,
            )
        )

        sizes = (
            position_size_from_confidence(
                confidence,
                low_threshold,
                high_threshold,
                low_bet,
                high_bet,
            )
        )

        trades = (
            run_sized_backtest(
                bars,
                validation,
                signals,
                sizes,
                tp,
                sl,
                end_time=val_end,
            )
        )

        if (
            trades.empty
            or
            len(trades)
            <
            MIN_VALIDATION_TRADES
        ):
            continue

        stats = (
            sized_strategy_stats(
                trades
            )
        )

        avg_return = (
            stats[
                "avg_return"
            ]
        )

        max_dd = (
            abs(
                stats[
                    "max_dd"
                ]
            )
        )

        n = (
            stats[
                "trades"
            ]
        )

        # DDが大きすぎる設定を罰する
        penalty = (
            1
            +
            5
            * max_dd
        )

        score = (
            avg_return
            *
            np.sqrt(n)
            /
            penalty
        )

        if (
            score
            >
            best_score
        ):

            best_score = (
                score
            )

            best = {
                "move_threshold":
                    move_threshold,

                "low_threshold":
                    low_threshold,

                "high_threshold":
                    high_threshold,

                "low_bet":
                    low_bet,

                "high_bet":
                    high_bet,

                "tp":
                    tp,

                "sl":
                    sl,

                "validation_trades":
                    n,

                "validation_avg_return":
                    avg_return,

                "validation_pf":
                    stats[
                        "profit_factor"
                    ],

                "validation_dd":
                    stats[
                        "max_dd"
                    ],

                "score":
                    score,
            }

    return best


# ============================================================
# 10. Walk-Forward
# ============================================================

fold_rows = []

all_test_trades = []

all_calibration = []


for fold in outer_folds(
    data
):

    print()
    print(
        "=================================="
    )

    print(
        f"Fold {fold.number}"
    )

    print(
        "=================================="
    )

    if (
        len(fold.train) < 500
        or
        len(fold.validation) == 0
        or
        len(fold.test) == 0
    ):

        print(
            "データ不足でskip"
        )

        continue

    # --------------------------------------------------------
    # Validationモデル
    # --------------------------------------------------------

    validation_model = (
        fit_base_models(
            fold.core,
            trees=300,
        )
    )

    if (
        validation_model
        is None
    ):

        print(
            "モデル学習不可"
        )

        continue

    validation_prob = (
        predict_base_models(
            validation_model,
            fold.validation,
        )
    )

    # --------------------------------------------------------
    # Validationで設定決定
    # --------------------------------------------------------

    print(
        "Validationで利益設定探索中..."
    )

    setting = (
        choose_profit_setting(
            bars,
            fold.validation,
            validation_prob,
        )
    )

    if (
        setting
        is None
    ):

        print(
            "設定を選べませんでした"
        )

        continue

    print(
        "選択設定:"
    )

    print(
        setting
    )

    # --------------------------------------------------------
    # Test直前まで再学習
    # --------------------------------------------------------

    final_model = (
        fit_base_models(
            fold.train,
            trees=400,
        )
    )

    if (
        final_model
        is None
    ):

        continue

    (
        p_move,
        p_up,
        p_down,
    ) = (
        predict_base_models(
            final_model,
            fold.test,
        )
    )

    # --------------------------------------------------------
    # Calibration
    # --------------------------------------------------------

    calibration = (
        confidence_calibration_table(
            p_up,
            p_down,
            fold.test[
                "future_return"
            ].values,
        )
    )

    calibration[
        "fold"
    ] = (
        fold.number
    )

    all_calibration.append(
        calibration
    )

    # --------------------------------------------------------
    # Test
    # --------------------------------------------------------

    (
        signals,
        confidence,
    ) = (
        make_confidence_signal(
            p_move,
            p_up,
            p_down,
            setting[
                "move_threshold"
            ],
            setting[
                "low_threshold"
            ],
            setting[
                "low_threshold"
            ],
        )
    )

    sizes = (
        position_size_from_confidence(
            confidence,
            setting[
                "low_threshold"
            ],
            setting[
                "high_threshold"
            ],
            setting[
                "low_bet"
            ],
            setting[
                "high_bet"
            ],
        )
    )

    test_end = (
        fold.test.index[-1]
        +
        pd.Timedelta(
            minutes=5
        )
    )

    trades = (
        run_sized_backtest(
            bars,
            fold.test,
            signals,
            sizes,
            setting[
                "tp"
            ],
            setting[
                "sl"
            ],
            end_time=test_end,
        )
    )

    if (
        not trades.empty
    ):

        trades[
            "fold"
        ] = (
            fold.number
        )

        trades[
            "confidence"
        ] = (
            confidence[
                signals != 0
            ][:len(trades)]
            if len(trades)
            else []
        )

        all_test_trades.append(
            trades
        )

    stats = (
        sized_strategy_stats(
            trades
        )
    )

    row = {
        "fold":
            fold.number,

        **setting,

        **{
            f"test_{k}":
                v
            for k, v
            in stats.items()
        },
    }

    fold_rows.append(
        row
    )

    print(
        "Test:",
        stats
    )


# ============================================================
# 11. 結合
# ============================================================

fold_results = (
    pd.DataFrame(
        fold_rows
    )
)

if (
    all_test_trades
):

    all_trades = (
        pd.concat(
            all_test_trades,
            ignore_index=True,
        )
    )

else:

    all_trades = (
        pd.DataFrame()
    )

if (
    all_calibration
):

    calibration_table = (
        pd.concat(
            all_calibration,
            ignore_index=True,
        )
    )

else:

    calibration_table = (
        pd.DataFrame()
    )


# ============================================================
# 12. 最終成績
# ============================================================

print()
print(
    "=================================="
)

print(
    "CONFIDENCE SIZING 最終結果"
)

print(
    "=================================="
)


final_stats = (
    sized_strategy_stats(
        all_trades
    )
)


for key, value in (
    final_stats.items()
):

    print(
        key,
        ":",
        value
    )


# ============================================================
# 13. Confidence別成績
# ============================================================

if (
    not all_trades.empty
):

    bins = [
        0.50,
        0.55,
        0.60,
        0.65,
        0.70,
        0.75,
        0.80,
        0.90,
        1.01,
    ]

    labels = [
        "50-55",
        "55-60",
        "60-65",
        "65-70",
        "70-75",
        "75-80",
        "80-90",
        "90-100",
    ]

    all_trades[
        "confidence_band"
    ] = (
        pd.cut(
            all_trades[
                "confidence"
            ],
            bins=bins,
            labels=labels,
            right=False,
        )
    )

    confidence_rows = []

    for band in labels:

        subset = (
            all_trades.loc[
                all_trades[
                    "confidence_band"
                ]
                == band
            ]
        )

        stats = (
            sized_strategy_stats(
                subset
            )
        )

        confidence_rows.append(
            {
                "confidence_band":
                    band,

                **stats
            }
        )

    confidence_results = (
        pd.DataFrame(
            confidence_rows
        )
    )

    print()
    print(
        "=================================="
    )

    print(
        "Confidence帯別"
    )

    print(
        "=================================="
    )

    print(
        confidence_results.to_string(
            index=False
        )
    )


# ============================================================
# 14. Fold安定性
# ============================================================

print()
print(
    "=================================="
)

print(
    "Fold安定性"
)

print(
    "=================================="
)


if (
    not fold_results.empty
):

    positive = (
        fold_results[
            "test_avg_return"
        ]
        > 0
    ).sum()

    pf_good = (
        fold_results[
            "test_profit_factor"
        ]
        > 1
    ).sum()

    print(
        "評価Fold:",
        len(
            fold_results
        )
    )

    print(
        "平均リターンプラス:",
        positive,
        "/",
        len(
            fold_results
        )
    )

    print(
        "PF > 1:",
        pf_good,
        "/",
        len(
            fold_results
        )
    )


# ============================================================
# 15. 資産曲線
# ============================================================

if (
    not all_trades.empty
):

    ordered = (
        all_trades
        .sort_values(
            "exit_time"
        )
        .copy()
    )

    ordered[
        "equity"
    ] = (
        START_CAPITAL
        *
        (
            1
            +
            ordered[
                "sized_return"
            ]
        ).cumprod()
    )

    plt.figure(
        figsize=(
            10,
            5
        )
    )

    plt.plot(
        ordered[
            "exit_time"
        ],
        ordered[
            "equity"
        ]
    )

    plt.xlabel(
        "Time"
    )

    plt.ylabel(
        "Capital"
    )

    plt.title(
        "Confidence-sized equity"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# 16. Calibration表示
# ============================================================

if (
    not calibration_table.empty
):

    calibration_summary = (
        calibration_table
        .groupby(
            "group",
            observed=True
        )
        .agg(
            count=(
                "count",
                "sum"
            ),

            mean_confidence=(
                "mean_confidence",
                "mean"
            ),

            actual_accuracy=(
                "actual_accuracy",
                "mean"
            ),

            avg_future_return=(
                "avg_future_return",
                "mean"
            ),
        )
        .reset_index()
    )

    print()
    print(
        "=================================="
    )

    print(
        "確率Calibration"
    )

    print(
        "=================================="
    )

    print(
        calibration_summary.to_string(
            index=False
        )
    )


# ============================================================
# 17. 保存
# ============================================================

output_dir = (
    Path.cwd()
    /
    "confidence_sizing_experiment"
)

output_dir.mkdir(
    exist_ok=True
)


fold_results.to_csv(
    output_dir
    / "fold_results.csv",
    index=False,
)

if (
    not all_trades.empty
):

    all_trades.to_csv(
        output_dir
        / "trades.csv",
        index=False,
    )

if (
    not calibration_table.empty
):

    calibration_table.to_csv(
        output_dir
        / "calibration.csv",
        index=False,
    )

if (
    "confidence_results"
    in globals()
):

    confidence_results.to_csv(
        output_dir
        / "confidence_bands.csv",
        index=False,
    )


print()
print(
    "=================================="
)

print(
    "実験完了"
)

print(
    "=================================="
)

print(
    "保存先:"
)

print(
    output_dir.resolve()
)

print()
print(
    "次に確認するもの"
)

print(
    "1. Confidence帯が高いほどaccuracyが上がるか"
)

print(
    "2. Confidence帯が高いほど平均利益が上がるか"
)

print(
    "3. 高BETを入れてPF/Growthが改善したか"
)

print(
    "4. 最大DDが悪化していないか"
)

print(
    "5. 複数Foldでプラスか"
)

## 元のセル index 23


In [ ]:
# ============================================================
# CALIBRATED CONFIDENCE SIZING + TP/SL + WALK-FORWARD
#
# 目的
# 1. RandomForestの生確率をValidationで校正する
# 2. 実際に約定した取引とconfidenceを正確に紐付ける
# 3. 校正後confidenceに応じてBETサイズを変更する
# 4. TP/SLもValidationで選ぶ
# 5. Testでは設定を一切変更しない
#
# 前提:
# 以前のセルで以下が定義済み
#
# load_bars
# prepare_data
# outer_folds
# fit_base_models
# predict_base_models
# simulate_trade
# strategy_stats
# HORIZON_BARS
# ============================================================


from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression


# ============================================================
# 1. 実験設定
# ============================================================

START_CAPITAL = 10000

MOVE_THRESHOLDS = [
    0.55,
    0.60,
    0.65,
    0.70,
]

LOW_CONF_THRESHOLDS = [
    0.52,
    0.55,
    0.58,
    0.60,
]

HIGH_CONF_THRESHOLDS = [
    0.62,
    0.65,
    0.70,
    0.75,
]

LOW_BETS = [
    0.25,
    0.50,
    0.75,
]

HIGH_BETS = [
    0.75,
    1.00,
    1.50,
]

TP_VALUES = [
    0.0003,
    0.0005,
    0.0008,
    0.0010,
    0.0015,
]

SL_VALUES = [
    0.0003,
    0.0005,
    0.0007,
    0.0010,
    0.0015,
]

MIN_VALIDATION_TRADES = 20


# ============================================================
# 2. 最新CSV取得
# ============================================================

def find_latest_snapshot():

    root = (
        Path.cwd()
        / "fx_experiment_runs"
    )

    files = list(
        root.glob(
            "*/usdjpy_5m.csv"
        )
    )

    if not files:
        raise FileNotFoundError(
            "usdjpy_5m.csv が見つかりません"
        )

    return max(
        files,
        key=lambda p: p.stat().st_mtime
    )


csv_path = find_latest_snapshot()

print("使用CSV:")
print(csv_path)


# ============================================================
# 3. データ読み込み
# ============================================================

bars = load_bars(
    csv_path
)

data = prepare_data(
    bars
)

print()
print("5分足:", len(bars))
print("使用可能データ:", len(data))


# ============================================================
# 4. 確率校正器
#
# 今回は
# isotonic
# logistic
# の2種類を比較
# ============================================================

def fit_probability_calibrator(
    raw_prob,
    target,
    method="isotonic",
):

    raw_prob = np.asarray(
        raw_prob,
        dtype=float
    )

    target = np.asarray(
        target,
        dtype=int
    )

    if len(
        np.unique(
            target
        )
    ) < 2:

        return None

    if method == "isotonic":

        model = IsotonicRegression(
            out_of_bounds="clip"
        )

        model.fit(
            raw_prob,
            target
        )

        return (
            "isotonic",
            model
        )

    elif method == "logistic":

        model = LogisticRegression(
            random_state=42
        )

        model.fit(
            raw_prob.reshape(
                -1,
                1
            ),
            target
        )

        return (
            "logistic",
            model
        )

    else:

        raise ValueError(
            "method must be isotonic or logistic"
        )


def apply_calibrator(
    calibrator,
    raw_prob,
):

    if calibrator is None:

        return np.asarray(
            raw_prob
        )

    method, model = calibrator

    raw_prob = np.asarray(
        raw_prob
    )

    if method == "isotonic":

        return model.predict(
            raw_prob
        )

    elif method == "logistic":

        return (
            model.predict_proba(
                raw_prob.reshape(
                    -1,
                    1
                )
            )[:, 1]
        )

    raise ValueError(
        "Unknown calibrator"
    )


# ============================================================
# 5. Direction確率をBUY/SELL用に校正
#
# p_up → P(actual UP)
# p_down → P(actual DOWN)
# ============================================================

def fit_direction_calibrators(
    validation_frame,
    p_up,
    p_down,
    method,
):

    up_target = (
        validation_frame[
            "future_return"
        ].values
        > 0
    ).astype(int)

    down_target = (
        validation_frame[
            "future_return"
        ].values
        < 0
    ).astype(int)

    up_calibrator = (
        fit_probability_calibrator(
            p_up,
            up_target,
            method=method
        )
    )

    down_calibrator = (
        fit_probability_calibrator(
            p_down,
            down_target,
            method=method
        )
    )

    return (
        up_calibrator,
        down_calibrator,
    )


# ============================================================
# 6. 校正後シグナル
# ============================================================

def make_calibrated_signals(
    p_move,
    calibrated_up,
    calibrated_down,
    move_threshold,
    low_threshold,
):

    signals = np.zeros(
        len(p_move)
    )

    buy = (
        (p_move >= move_threshold)
        &
        (calibrated_up >= low_threshold)
        &
        (calibrated_up > calibrated_down)
    )

    sell = (
        (p_move >= move_threshold)
        &
        (calibrated_down >= low_threshold)
        &
        (calibrated_down > calibrated_up)
    )

    signals[buy] = 1
    signals[sell] = -1

    confidence = np.maximum(
        calibrated_up,
        calibrated_down
    )

    return (
        signals,
        confidence
    )


# ============================================================
# 7. BET倍率
# ============================================================

def make_position_sizes(
    confidence,
    low_threshold,
    high_threshold,
    low_bet,
    high_bet,
):

    size = np.zeros(
        len(confidence)
    )

    medium = (
        (confidence >= low_threshold)
        &
        (confidence < high_threshold)
    )

    high = (
        confidence >= high_threshold
    )

    size[medium] = low_bet
    size[high] = high_bet

    return size


# ============================================================
# 8. 正確な約定取引との紐付け付きBacktest
#
# 前回の問題点:
# confidence[signals != 0][:len(trades)]
# だと保有中に無視されたシグナルとのズレが出る
#
# 今回は各tradeを生成したsignal_timeから
# confidence等を直接保存する
# ============================================================

def run_precise_sized_backtest(
    bars,
    frame,
    signals,
    sizes,
    confidence,
    p_move,
    calibrated_up,
    calibrated_down,
    tp,
    sl,
    *,
    end_time=None,
):

    records = []

    next_signal_position = -1

    for i, time in enumerate(
        frame.index
    ):

        signal = signals[i]
        size = sizes[i]

        if (
            signal == 0
            or
            size <= 0
        ):
            continue

        position = (
            bars.index
            .get_loc(
                time
            )
        )

        if (
            position
            <
            next_signal_position
        ):
            continue

        trade = simulate_trade(
            bars,
            time,
            (
                "BUY"
                if signal == 1
                else "SELL"
            ),
            tp,
            sl,
            end_time=end_time,
        )

        if trade is None:
            continue

        # ------------------------------------
        # ここで同じsignal_timeの情報を保存
        # これでconfidenceのズレを防ぐ
        # ------------------------------------

        trade[
            "position_size"
        ] = size

        trade[
            "confidence"
        ] = confidence[i]

        trade[
            "p_move"
        ] = p_move[i]

        trade[
            "calibrated_up"
        ] = calibrated_up[i]

        trade[
            "calibrated_down"
        ] = calibrated_down[i]

        trade[
            "sized_return"
        ] = (
            trade[
                "net_return"
            ]
            *
            size
        )

        records.append(
            trade
        )

        next_signal_position = (
            position
            +
            HORIZON_BARS
        )

    return pd.DataFrame(
        records
    )


# ============================================================
# 9. sized stats
# ============================================================

def sized_strategy_stats(
    trades
):

    if (
        trades is None
        or
        trades.empty
    ):

        return {
            "trades":
                0,

            "win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "profit_factor":
                np.nan,

            "max_dd":
                np.nan,

            "total_growth":
                0.0,

            "sharpe":
                np.nan,

            "end_capital":
                START_CAPITAL,
        }

    r = (
        trades[
            "sized_return"
        ]
        .astype(float)
        .to_numpy()
    )

    base = strategy_stats(
        r
    )

    if (
        len(r) > 1
        and
        np.std(r) > 0
    ):

        sharpe = (
            np.mean(r)
            /
            np.std(r)
            *
            np.sqrt(
                len(r)
            )
        )

    else:

        sharpe = np.nan

    end_capital = (
        START_CAPITAL
        *
        np.prod(
            1 + r
        )
    )

    base[
        "sharpe"
    ] = sharpe

    base[
        "end_capital"
    ] = end_capital

    return base


# ============================================================
# 10. Calibration評価
#
# Brier Score的に
# squared errorを見る
# ============================================================

def calibration_error(
    calibrated_prob,
    target,
):

    p = np.asarray(
        calibrated_prob
    )

    y = np.asarray(
        target
    )

    return np.mean(
        (
            p - y
        ) ** 2
    )


# ============================================================
# 11. Validationで
# 校正方法 + BET + TP/SL
# を選ぶ
# ============================================================

def choose_best_setting(
    bars,
    validation,
    raw_probabilities,
):

    p_move, raw_p_up, raw_p_down = (
        raw_probabilities
    )

    val_end = (
        validation.index[-1]
        +
        pd.Timedelta(
            minutes=5
        )
    )

    best = None
    best_score = -np.inf

    for calibration_method in [
        "isotonic",
        "logistic",
    ]:

        (
            up_calibrator,
            down_calibrator,
        ) = fit_direction_calibrators(
            validation,
            raw_p_up,
            raw_p_down,
            method=calibration_method,
        )

        calibrated_up = (
            apply_calibrator(
                up_calibrator,
                raw_p_up
            )
        )

        calibrated_down = (
            apply_calibrator(
                down_calibrator,
                raw_p_down
            )
        )

        for (
            move_threshold,
            low_threshold,
            high_threshold,
            low_bet,
            high_bet,
            tp,
            sl,
        ) in itertools.product(

            MOVE_THRESHOLDS,

            LOW_CONF_THRESHOLDS,

            HIGH_CONF_THRESHOLDS,

            LOW_BETS,

            HIGH_BETS,

            TP_VALUES,

            SL_VALUES,
        ):

            if (
                high_threshold
                <= low_threshold
            ):
                continue

            if (
                high_bet
                < low_bet
            ):
                continue

            (
                signals,
                confidence,
            ) = (
                make_calibrated_signals(
                    p_move,
                    calibrated_up,
                    calibrated_down,
                    move_threshold,
                    low_threshold,
                )
            )

            sizes = (
                make_position_sizes(
                    confidence,
                    low_threshold,
                    high_threshold,
                    low_bet,
                    high_bet,
                )
            )

            trades = (
                run_precise_sized_backtest(
                    bars,
                    validation,
                    signals,
                    sizes,
                    confidence,
                    p_move,
                    calibrated_up,
                    calibrated_down,
                    tp,
                    sl,
                    end_time=val_end,
                )
            )

            if (
                trades.empty
                or
                len(trades)
                <
                MIN_VALIDATION_TRADES
            ):
                continue

            stats = (
                sized_strategy_stats(
                    trades
                )
            )

            avg_return = (
                stats[
                    "avg_return"
                ]
            )

            max_dd = abs(
                stats[
                    "max_dd"
                ]
            )

            n = (
                stats[
                    "trades"
                ]
            )

            # ------------------------------------
            # 利益重視だが、
            # 取引数とDDも少し考慮
            # ------------------------------------

            score = (
                avg_return
                *
                np.sqrt(n)
                /
                (
                    1
                    +
                    5
                    * max_dd
                )
            )

            if (
                score
                >
                best_score
            ):

                best_score = score

                best = {
                    "calibration_method":
                        calibration_method,

                    "move_threshold":
                        move_threshold,

                    "low_threshold":
                        low_threshold,

                    "high_threshold":
                        high_threshold,

                    "low_bet":
                        low_bet,

                    "high_bet":
                        high_bet,

                    "tp":
                        tp,

                    "sl":
                        sl,

                    "validation_trades":
                        n,

                    "validation_avg_return":
                        avg_return,

                    "validation_pf":
                        stats[
                            "profit_factor"
                        ],

                    "validation_dd":
                        stats[
                            "max_dd"
                        ],

                    "score":
                        score,
                }

    return best


# ============================================================
# 12. Walk-Forward
# ============================================================

fold_rows = []

all_test_trades = []

all_calibration_rows = []


for fold in outer_folds(
    data
):

    print()
    print(
        "===================================="
    )

    print(
        f"Fold {fold.number}"
    )

    print(
        "===================================="
    )

    if (
        len(fold.train) < 500
        or
        len(fold.validation) == 0
        or
        len(fold.test) == 0
    ):

        print(
            "データ不足でskip"
        )

        continue

    # --------------------------------------------------------
    # Validation用Baseモデル
    # --------------------------------------------------------

    validation_model = (
        fit_base_models(
            fold.core,
            trees=300,
        )
    )

    if (
        validation_model
        is None
    ):

        print(
            "Validationモデル学習不可"
        )

        continue

    validation_raw_prob = (
        predict_base_models(
            validation_model,
            fold.validation,
        )
    )

    # --------------------------------------------------------
    # Validationで
    # calibration + BET + TP/SLを選択
    # --------------------------------------------------------

    print(
        "Validationで設定探索中..."
    )

    setting = (
        choose_best_setting(
            bars,
            fold.validation,
            validation_raw_prob,
        )
    )

    if (
        setting
        is None
    ):

        print(
            "設定を選べませんでした"
        )

        continue

    print(
        "選択設定:"
    )

    print(
        setting
    )

    # --------------------------------------------------------
    # Final Base
    # --------------------------------------------------------

    final_base = (
        fit_base_models(
            fold.train,
            trees=400,
        )
    )

    if (
        final_base
        is None
    ):

        continue

    # --------------------------------------------------------
    # Calibrationモデルは
    # Validationデータだけでfit
    # Testにはfitしない
    # --------------------------------------------------------

    (
        val_p_move,
        val_raw_up,
        val_raw_down,
    ) = (
        validation_raw_prob
    )

    (
        up_calibrator,
        down_calibrator,
    ) = fit_direction_calibrators(
        fold.validation,
        val_raw_up,
        val_raw_down,
        method=
            setting[
                "calibration_method"
            ],
    )

    # --------------------------------------------------------
    # Test予測
    # --------------------------------------------------------

    (
        test_p_move,
        test_raw_up,
        test_raw_down,
    ) = (
        predict_base_models(
            final_base,
            fold.test,
        )
    )

    calibrated_up = (
        apply_calibrator(
            up_calibrator,
            test_raw_up
        )
    )

    calibrated_down = (
        apply_calibrator(
            down_calibrator,
            test_raw_down
        )
    )

    # --------------------------------------------------------
    # Calibration診断用保存
    # --------------------------------------------------------

    actual_up = (
        fold.test[
            "future_return"
        ].values
        > 0
    ).astype(int)

    actual_down = (
        fold.test[
            "future_return"
        ].values
        < 0
    ).astype(int)

    for i, time in enumerate(
        fold.test.index
    ):

        all_calibration_rows.append(
            {
                "fold":
                    fold.number,

                "time":
                    time,

                "raw_p_up":
                    test_raw_up[i],

                "raw_p_down":
                    test_raw_down[i],

                "calibrated_up":
                    calibrated_up[i],

                "calibrated_down":
                    calibrated_down[i],

                "actual_up":
                    actual_up[i],

                "actual_down":
                    actual_down[i],

                "future_return":
                    fold.test[
                        "future_return"
                    ].iloc[i],
            }
        )

    # --------------------------------------------------------
    # Test signal
    # --------------------------------------------------------

    (
        signals,
        confidence,
    ) = (
        make_calibrated_signals(
            test_p_move,
            calibrated_up,
            calibrated_down,
            setting[
                "move_threshold"
            ],
            setting[
                "low_threshold"
            ],
        )
    )

    sizes = (
        make_position_sizes(
            confidence,
            setting[
                "low_threshold"
            ],
            setting[
                "high_threshold"
            ],
            setting[
                "low_bet"
            ],
            setting[
                "high_bet"
            ],
        )
    )

    test_end = (
        fold.test.index[-1]
        +
        pd.Timedelta(
            minutes=5
        )
    )

    trades = (
        run_precise_sized_backtest(
            bars,
            fold.test,
            signals,
            sizes,
            confidence,
            test_p_move,
            calibrated_up,
            calibrated_down,
            setting[
                "tp"
            ],
            setting[
                "sl"
            ],
            end_time=test_end,
        )
    )

    if (
        not trades.empty
    ):

        trades[
            "fold"
        ] = (
            fold.number
        )

        all_test_trades.append(
            trades
        )

    stats = (
        sized_strategy_stats(
            trades
        )
    )

    row = {
        "fold":
            fold.number,

        **setting,

        **{
            f"test_{k}":
                v
            for k, v
            in stats.items()
        },
    }

    fold_rows.append(
        row
    )

    print(
        "Test:"
    )

    print(
        stats
    )


# ============================================================
# 13. 結合
# ============================================================

fold_results = (
    pd.DataFrame(
        fold_rows
    )
)

if (
    all_test_trades
):

    all_trades = (
        pd.concat(
            all_test_trades,
            ignore_index=True,
        )
    )

else:

    all_trades = (
        pd.DataFrame()
    )


calibration_df = (
    pd.DataFrame(
        all_calibration_rows
    )
)


# ============================================================
# 14. 最終成績
# ============================================================

print()
print(
    "===================================="
)

print(
    "CALIBRATED SIZING 最終結果"
)

print(
    "===================================="
)


final_stats = (
    sized_strategy_stats(
        all_trades
    )
)


for k, v in (
    final_stats.items()
):

    print(
        k,
        ":",
        v
    )


# ============================================================
# 15. Fold安定性
# ============================================================

print()
print(
    "===================================="
)

print(
    "Fold安定性"
)

print(
    "===================================="
)


if (
    not fold_results.empty
):

    positive = (
        fold_results[
            "test_avg_return"
        ]
        > 0
    ).sum()

    pf_good = (
        fold_results[
            "test_profit_factor"
        ]
        > 1
    ).sum()

    print(
        "評価Fold:",
        len(
            fold_results
        )
    )

    print(
        "平均リターンプラス:",
        positive,
        "/",
        len(
            fold_results
        )
    )

    print(
        "PF > 1:",
        pf_good,
        "/",
        len(
            fold_results
        )
    )


# ============================================================
# 16. 正確なConfidence帯別
# ============================================================

if (
    not all_trades.empty
):

    bins = [
        0.50,
        0.55,
        0.60,
        0.65,
        0.70,
        0.75,
        0.80,
        0.90,
        1.01,
    ]

    labels = [
        "50-55",
        "55-60",
        "60-65",
        "65-70",
        "70-75",
        "75-80",
        "80-90",
        "90-100",
    ]

    all_trades[
        "confidence_band"
    ] = (
        pd.cut(
            all_trades[
                "confidence"
            ],
            bins=bins,
            labels=labels,
            right=False,
        )
    )

    band_rows = []

    for band in labels:

        subset = (
            all_trades.loc[
                all_trades[
                    "confidence_band"
                ]
                == band
            ]
        )

        stats = (
            sized_strategy_stats(
                subset
            )
        )

        band_rows.append(
            {
                "confidence_band":
                    band,

                **stats
            }
        )

    band_results = (
        pd.DataFrame(
            band_rows
        )
    )

    print()
    print(
        "===================================="
    )

    print(
        "正確なConfidence帯別"
    )

    print(
        "===================================="
    )

    print(
        band_results.to_string(
            index=False
        )
    )


# ============================================================
# 17. Calibration診断
# ============================================================

if (
    not calibration_df.empty
):

    calibration_df[
        "confidence"
    ] = (
        calibration_df[
            [
                "calibrated_up",
                "calibrated_down",
            ]
        ]
        .max(axis=1)
    )

    calibration_df[
        "predicted_up"
    ] = (
        calibration_df[
            "calibrated_up"
        ]
        >
        calibration_df[
            "calibrated_down"
        ]
    )

    calibration_df[
        "correct"
    ] = np.where(
        calibration_df[
            "predicted_up"
        ],
        calibration_df[
            "actual_up"
        ],
        calibration_df[
            "actual_down"
        ],
    )

    bins = [
        0.50,
        0.55,
        0.60,
        0.65,
        0.70,
        0.75,
        0.80,
        0.90,
        1.01,
    ]

    labels = [
        "50-55",
        "55-60",
        "60-65",
        "65-70",
        "70-75",
        "75-80",
        "80-90",
        "90-100",
    ]

    calibration_df[
        "band"
    ] = (
        pd.cut(
            calibration_df[
                "confidence"
            ],
            bins=bins,
            labels=labels,
            right=False,
        )
    )

    calibration_summary = (
        calibration_df
        .groupby(
            "band",
            observed=True
        )
        .agg(
            count=(
                "correct",
                "size"
            ),

            mean_confidence=(
                "confidence",
                "mean"
            ),

            actual_accuracy=(
                "correct",
                "mean"
            ),

            avg_future_return=(
                "future_return",
                "mean"
            ),
        )
        .reset_index()
    )

    print()
    print(
        "===================================="
    )

    print(
        "校正後Calibration"
    )

    print(
        "===================================="
    )

    print(
        calibration_summary.to_string(
            index=False
        )
    )


# ============================================================
# 18. Raw vs Calibrated Brier Score
# ============================================================

if (
    not calibration_df.empty
):

    raw_brier_up = (
        calibration_error(
            calibration_df[
                "raw_p_up"
            ],
            calibration_df[
                "actual_up"
            ],
        )
    )

    calibrated_brier_up = (
        calibration_error(
            calibration_df[
                "calibrated_up"
            ],
            calibration_df[
                "actual_up"
            ],
        )
    )

    raw_brier_down = (
        calibration_error(
            calibration_df[
                "raw_p_down"
            ],
            calibration_df[
                "actual_down"
            ],
        )
    )

    calibrated_brier_down = (
        calibration_error(
            calibration_df[
                "calibrated_down"
            ],
            calibration_df[
                "actual_down"
            ],
        )
    )

    print()
    print(
        "===================================="
    )

    print(
        "Brier Score"
    )

    print(
        "===================================="
    )

    print(
        "UP raw:",
        raw_brier_up
    )

    print(
        "UP calibrated:",
        calibrated_brier_up
    )

    print(
        "DOWN raw:",
        raw_brier_down
    )

    print(
        "DOWN calibrated:",
        calibrated_brier_down
    )


# ============================================================
# 19. 資産曲線
# ============================================================

if (
    not all_trades.empty
):

    ordered = (
        all_trades
        .sort_values(
            "exit_time"
        )
        .copy()
    )

    ordered[
        "equity"
    ] = (
        START_CAPITAL
        *
        (
            1
            +
            ordered[
                "sized_return"
            ]
        ).cumprod()
    )

    plt.figure(
        figsize=(
            10,
            5
        )
    )

    plt.plot(
        ordered[
            "exit_time"
        ],
        ordered[
            "equity"
        ]
    )

    plt.xlabel(
        "Time"
    )

    plt.ylabel(
        "Capital"
    )

    plt.title(
        "Calibrated Confidence Sized Equity"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# 20. Confidence vs accuracy graph
# ============================================================

if (
    "calibration_summary"
    in globals()
):

    temp = (
        calibration_summary
        .dropna()
        .copy()
    )

    plt.figure(
        figsize=(
            7,
            5
        )
    )

    plt.plot(
        temp[
            "mean_confidence"
        ],
        temp[
            "actual_accuracy"
        ],
        marker="o"
    )

    plt.plot(
        [
            0.5,
            1.0
        ],
        [
            0.5,
            1.0
        ],
        linestyle="--"
    )

    plt.xlabel(
        "Calibrated confidence"
    )

    plt.ylabel(
        "Actual accuracy"
    )

    plt.title(
        "Calibration Curve"
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# 21. 保存
# ============================================================

output_dir = (
    Path.cwd()
    /
    "calibrated_confidence_experiment"
)

output_dir.mkdir(
    exist_ok=True
)


fold_results.to_csv(
    output_dir
    / "fold_results.csv",
    index=False,
)

if (
    not all_trades.empty
):

    all_trades.to_csv(
        output_dir
        / "trades.csv",
        index=False,
    )

if (
    not calibration_df.empty
):

    calibration_df.to_csv(
        output_dir
        / "calibration_rows.csv",
        index=False,
    )

if (
    "band_results"
    in globals()
):

    band_results.to_csv(
        output_dir
        / "confidence_bands.csv",
        index=False,
    )

if (
    "calibration_summary"
    in globals()
):

    calibration_summary.to_csv(
        output_dir
        / "calibration_summary.csv",
        index=False,
    )


print()
print(
    "===================================="
)

print(
    "実験完了"
)

print(
    "===================================="
)

print(
    "保存先:"
)

print(
    output_dir.resolve()
)

print()
print(
    "今回見るポイント"
)

print(
    "1. Calibration後、Brier Scoreが改善したか"
)

print(
    "2. Confidenceが高いほどactual_accuracyが上がるか"
)

print(
    "3. Confidenceが高いほど平均リターンが上がるか"
)

print(
    "4. 高BETを入れてPF/Growthが改善したか"
)

print(
    "5. 最大DDが悪化していないか"
)

print(
    "6. 複数Foldでプラスか"
)